# AGENTS_030 — Change Impact Agent (Hackathon Demo)

This notebook:
1. Clones **your fork** (`rajipsv/TheRock`, branch `feature/change-impact-agent`)
2. Installs Python dependencies
3. Runs unit tests (`pytest`)
4. Analyzes real upstream PRs on `ROCm/TheRock`
5. Generates executive summaries and displays HTML reports (including **Topology warnings** — always shown, even when empty)

**Pipeline:** `analyze.py` is fully deterministic (topology graph + `topology_audit.py`). `summarize.py` uses `--backend template` by default; optional LLM backends apply validation guardrails.

**Secrets:** paste `GITHUB_TOKEN` in `agents/change-impact-agent/.env` (gitignored) — never hardcode tokens in this notebook.

**Out of scope:** ARVIL integration, cross-repo pattern scanning (see `SCOPE.md` in the agent folder).

In [2]:
import os
import subprocess
import sys
from pathlib import Path

# --- Configuration (edit if needed) ---
FORK_REPO = "https://github.com/rajipsv/TheRock.git"
BRANCH = "feature/change-impact-agent"
CLONE_DIR = Path(os.environ.get("THEROCK_CLONE", Path.home() / "TheRock-fork-demo"))
UPSTREAM_REPO = "ROCm/TheRock"
# PRs used in the hackathon demo (upstream ROCm/TheRock)
DEMO_PRS = [5572, 5688, 5480, 5718]

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")

if GITHUB_TOKEN:
    print("GITHUB_TOKEN set — superrepo component detection enabled")
else:
    print("GITHUB_TOKEN not set — some PRs may have partial component lists (rate limits)")

print(f"Clone target: {CLONE_DIR}")

GITHUB_TOKEN set — superrepo component detection enabled
Clone target: C:\Users\Rajeswari\TheRock-fork-demo


## 1. Clone fork (skip if already cloned)

In [3]:
def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode, cmd, output=result.stdout, stderr=result.stderr
        )
    return result

if (CLONE_DIR / ".git").exists():
    print(f"Repo exists at {CLONE_DIR} — fetching branch {BRANCH}")
    run(["git", "fetch", "origin", BRANCH], cwd=CLONE_DIR)
    run(["git", "checkout", BRANCH], cwd=CLONE_DIR)
    run(["git", "pull", "origin", BRANCH], cwd=CLONE_DIR, check=False)
    run(["git", "fetch", "origin", BRANCH, "--depth", "80"], cwd=CLONE_DIR, check=False)
    run([
        "git", "fetch",
        f"https://github.com/{UPSTREAM_REPO}.git",
        "main:upstream-main", "--depth", "200",
    ], cwd=CLONE_DIR, check=False)
else:
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    run([
        "git", "clone",
        "--branch", BRANCH,
        "--depth", "80",
        FORK_REPO,
        str(CLONE_DIR),
    ])
    run([
        "git", "fetch",
        f"https://github.com/{UPSTREAM_REPO}.git",
        "main:upstream-main", "--depth", "200",
    ], cwd=CLONE_DIR, check=False)

REPO = CLONE_DIR.resolve()
AGENT = REPO / "agents" / "change-impact-agent"
OUT = AGENT / "out"
# Load agents/change-impact-agent/.env if you created it (GITHUB_TOKEN)
sys.path.insert(0, str(AGENT))
from env_loader import load_agent_env
load_agent_env()
if os.environ.get("GITHUB_TOKEN"):
    print("Loaded GITHUB_TOKEN from .env or environment")
else:
    print(f"No GITHUB_TOKEN — copy .env to {AGENT / '.env'} (see .env.example)")

import json


def summarize_cli_supports_validation() -> bool:
    help_text = subprocess.run(
        [sys.executable, str(AGENT / "summarize.py"), "--help"],
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    return "--validate-llm" in (help_text.stdout + help_text.stderr)


def configure_vllm_client_env(
    base_url: str = "http://localhost:8000/v1",
    api_key: str = "abc-123",
    model: str = "Qwen3-30B-A3B",
) -> tuple[str, str, str]:
    """AMD AI Agents 101 vLLM workshop env pattern (OpenAI-compatible endpoint)."""
    os.environ["VLLM_BASE_URL"] = base_url
    os.environ["BASE_URL"] = base_url
    os.environ["OPENAI_API_KEY"] = api_key
    os.environ["VLLM_MODEL"] = model
    os.environ["SUMMARY_LLM_BACKEND"] = "vllm"
    return base_url, api_key, model


def verify_vllm_server(base_url: str, api_key: str) -> bool:
    import urllib.error
    import urllib.request

    url = f"{base_url.rstrip('/')}/models"
    req = urllib.request.Request(
        url,
        headers={"Authorization": f"Bearer {api_key}"},
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            body = resp.read().decode("utf-8", errors="replace")
            print(body[:800])
            if len(body) > 800:
                print("...")
            return True
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        print(f"vLLM not reachable at {url}: {exc}")
        return False


def print_topology_warnings(report_path: Path) -> None:
    """Mirror report.html / executive_summary topology section."""
    if not report_path.is_file():
        return
    report = json.loads(report_path.read_text(encoding="utf-8"))
    warnings = report.get("topology_warnings") or []
    print("Topology warnings:")
    if warnings:
        for warning in warnings:
            print(f"  - {warning}")
    else:
        print("  - No topology gaps detected for this change range.")


def summarize_pr_output(
    pr_out: Path,
    backend: str = "template",
    *,
    base_url: str | None = None,
    model: str | None = None,
    output_name: str = "executive_summary.md",
) -> str:
    """Run summarize.py (template or LLM backend with validation guardrails)."""
    output_path = pr_out / output_name
    cmd = [
        sys.executable,
        str(AGENT / "summarize.py"),
        "--backend",
        backend,
        "--input",
        str(pr_out / "report.json"),
        "--output",
        str(output_path),
    ]
    if base_url:
        cmd.extend(["--base-url", base_url])
    if model:
        cmd.extend(["--model", model])
    if backend != "template" and summarize_cli_supports_validation():
        cmd.extend(["--validate-llm", "--fallback-template"])
    run(cmd, cwd=REPO)
    print_topology_warnings(pr_out / "report.json")
    return output_path.read_text(encoding="utf-8")


def run_llm_executive_summary(
    pr_out: Path,
    *,
    backend: str = "vllm",
    base_url: str | None = None,
    model: str | None = None,
    output_name: str = "executive_summary_llm.md",
) -> str | None:
    """Generate LLM prose summary; keeps template executive_summary.md unchanged."""
    load_agent_env()
    report_path = pr_out / "report.json"
    if not report_path.is_file():
        print(f"No report at {report_path} — run section 5 first.")
        return None

    if backend == "openai":
        if not os.environ.get("OPENAI_API_KEY"):
            print("OPENAI_API_KEY not set — add to .env or skip this section.")
            return None
        base_url = base_url or "https://api.openai.com/v1"
        model = model or os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
    elif backend == "vllm":
        base_url = base_url or os.environ.get(
            "VLLM_BASE_URL", os.environ.get("BASE_URL", "http://localhost:8000/v1")
        )
        model = model or os.environ.get("VLLM_MODEL", "Qwen3-30B-A3B")
        os.environ.setdefault("OPENAI_API_KEY", "abc-123")
    else:
        raise ValueError(f"Unsupported LLM backend: {backend}")

    print(f"LLM summary: backend={backend}, model={model}, url={base_url}")
    try:
        return summarize_pr_output(
            pr_out,
            backend=backend,
            base_url=base_url,
            model=model,
            output_name=output_name,
        )
    except subprocess.CalledProcessError as exc:
        print(
            f"LLM summary failed (exit {exc.returncode}). "
            "Template executive_summary.md from summarize step is unchanged."
        )
        return None


print(f"TheRock root: {REPO}")
print(f"Agent: {AGENT}")

Repo exists at C:\Users\Rajeswari\TheRock-fork-demo — fetching branch feature/change-impact-agent
$ git fetch origin feature/change-impact-agent
From https://github.com/rajipsv/TheRock
 * branch            feature/change-impact-agent -> FETCH_HEAD

$ git checkout feature/change-impact-agent
Your branch is up to date with 'origin/feature/change-impact-agent'.

Already on 'feature/change-impact-agent'

$ git pull origin feature/change-impact-agent
Already up to date.

From https://github.com/rajipsv/TheRock
 * branch            feature/change-impact-agent -> FETCH_HEAD

$ git fetch origin feature/change-impact-agent --depth 80
From https://github.com/rajipsv/TheRock
 * branch            feature/change-impact-agent -> FETCH_HEAD

$ git fetch https://github.com/ROCm/TheRock.git main:upstream-main --depth 200
No GITHUB_TOKEN — copy .env to C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\.env (see .env.example)
TheRock root: C:\Users\Rajeswari\TheRock-fork-demo
Agent: C:\User

## 2. Install dependencies

In [4]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(AGENT / "requirements.txt")])
run([sys.executable, "-m", "pip", "install", "-q", "pytest"])

$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe -m pip install -q -r C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\requirements.txt

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe -m pip install -q pytest

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



CompletedProcess(args=['C:\\Users\\Rajeswari\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', '-m', 'pip', 'install', '-q', 'pytest'], returncode=0, stdout='', stderr='\n[notice] A new release of pip is available: 25.2 -> 26.1.2\n[notice] To update, run: python.exe -m pip install --upgrade pip\n')

## 3. Run unit tests

In [5]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "agents/change-impact-agent/tests/", "-q"],
    cwd=REPO,
)
if result.returncode != 0:
    raise RuntimeError("pytest failed — fix tests before demo")
print("All unit tests passed.")

All unit tests passed.


## 4. Analyze local range + executive summary

Step 1: `analyze.py` → `report.json` + `report.html`. Step 2: `summarize.py --backend template` → `executive_summary.md`.

Uses `HEAD~6..HEAD` on the checked-out branch (not `main~6..main` on shallow fork clones).

In [6]:
demo_out = OUT / "demo-main-range"
run([
    sys.executable, str(AGENT / "analyze.py"),
    "--start", "HEAD~6", "--end", "HEAD",
    "--output-dir", str(demo_out),
], cwd=REPO)
print(summarize_pr_output(demo_out))

$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\analyze.py --start HEAD~6 --end HEAD --output-dir C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\out\demo-main-range
Analyzing HEAD~6 -> HEAD ...
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 5045d587d3e2a525f717a027bb1f5a6fd4b81e1e -- base/half
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 5045d587d3e2a525f717a027bb1f5a6fd4b81e1e -- base/rocm-cmake
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 5045d587d3e2a525f717a027bb1f5a6fd4b81e1e -- compiler/amd-llvm
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 5045d587d3e2a525f717a027bb1f5a6fd4b81e1e -- compiler/hipify
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 5045d587d3e2a525f717a027bb1f5a6fd4b81e1e -- compiler/spirv-llvm-translator
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 5045d587d3e2a525f717a027bb1f5a

## 5. Analyze upstream PRs + executive summary (ROCm/TheRock)

Each PR: `analyze.py` then `summarize.py` (separate steps, same as CI). Section 1 fetches `upstream-main` for merge-base — re-run section 1 if PR analysis fails.

| PR | Type |
|----|------|
| 5572 | MIOpen GHA timeout 60→120 min |
| 5688 | hipDNN CI + artifact TOML |
| 5480 | OpenMPI CMake version file naming |
| 5718 | rocm-libraries superrepo bump |

In [7]:
UPSTREAM_GIT = f"https://github.com/{UPSTREAM_REPO}.git"

def ensure_upstream_main() -> None:
    """Fork single-branch clones have no local main — fetch upstream tip as upstream-main."""
    run([
        "git", "fetch", UPSTREAM_GIT, "main:upstream-main", "--depth", "200",
    ], cwd=REPO, check=False)

def analyze_supports_pr_flag() -> bool:
    help_text = subprocess.run(
        [sys.executable, str(AGENT / "analyze.py"), "--help"],
        cwd=REPO, capture_output=True, text=True,
    )
    return "--pr" in (help_text.stdout + help_text.stderr)

def analyze_upstream_pr(pr: int, pr_out: Path) -> int:
    """Analyze an upstream ROCm/TheRock PR from a fork clone."""
    ensure_upstream_main()
    if analyze_supports_pr_flag():
        return run([
            sys.executable, str(AGENT / "analyze.py"),
            "--pr", str(pr),
            "--full-manifest",
            "--output-dir", str(pr_out),
        ], cwd=REPO, check=False).returncode
    local_ref = f"pr-{pr}"
    run(["git", "fetch", UPSTREAM_GIT, f"pull/{pr}/head:{local_ref}"], cwd=REPO, check=False)
    return run([
        sys.executable, str(AGENT / "analyze.py"),
        "--end", local_ref,
        "--pr-base-ref", "upstream-main",
        "--output-dir", str(pr_out),
    ], cwd=REPO, check=False).returncode

pr_results = {}
for pr in DEMO_PRS:
    pr_out = OUT / f"pr-{pr}"
    print(f"\n{'='*60}\nPR #{pr}\n{'='*60}")
    rc = analyze_upstream_pr(pr, pr_out)
    if rc != 0:
        print(f"Warning: analyze failed for PR #{pr} (exit {rc})")
        continue
    summary = summarize_pr_output(pr_out)
    pr_results[pr] = summary
    # Print first 25 lines of each summary
    print("\n".join(summary.splitlines()[:25]))
    if len(summary.splitlines()) > 25:
        print("...")


PR #5572
$ git fetch https://github.com/ROCm/TheRock.git main:upstream-main --depth 200
$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\analyze.py --pr 5572 --full-manifest --output-dir C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\out\pr-5572
Analyzing 7b7b238e3e0c2e98436ea230b5114a5b6b946abf -> pr-5572 ...

Comparing commits: 7b7b238 -> c70f211

=== Getting submodules for START commit ===
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 7b7b238e3e0c2e98436ea230b5114a5b6b946abf -- base/half
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 7b7b238e3e0c2e98436ea230b5114a5b6b946abf -- base/rocm-cmake
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 7b7b238e3e0c2e98436ea230b5114a5b6b946abf -- compiler/amd-llvm
++ Exec [C:\Users\Rajeswari\TheRock-fork-demo]$ git ls-tree 7b7b238e3e0c2e98436ea230b5114a5b6b946abf -- compiler/hipify
++ Exec [C:\Users\Rajes

## 6. List open upstream PRs (optional, no analyze)

Requires `GITHUB_TOKEN` in `agents/change-impact-agent/.env` (GitHub rate-limits anonymous API calls).

In [8]:
load_agent_env()
if not os.environ.get("GITHUB_TOKEN"):
    print(
        "Warning: no GITHUB_TOKEN — create agents/change-impact-agent/.env "
        "(copy from .env.example) or section 6 may hit GitHub rate limits."
    )
rc = run([
    sys.executable, str(AGENT / "upstream_pr_scan.py"),
    "--max", "5",
], cwd=REPO, check=False).returncode
if rc != 0:
    print(f"upstream_pr_scan failed (exit {rc}). Add GITHUB_TOKEN to {AGENT / '.env'}")

$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\upstream_pr_scan.py --max 5
Fetching open PRs from ROCm/TheRock (max 5)...

Error: GitHub API rate limit — set GITHUB_TOKEN in agents/change-impact-agent/.env

upstream_pr_scan failed (exit 1). Add GITHUB_TOKEN to C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\.env


## 7. View HTML report (example: PR #5572)

In [9]:
from IPython.display import HTML

example_pr = 5718
html_path = OUT / f"pr-{example_pr}" / "report.html"
if html_path.exists():
    HTML(html_path.read_text(encoding="utf-8"))
else:
    print(f"No report at {html_path} — run section 5 first")

## 9. LLM executive summary — vLLM on MI300 (AMD workshop pattern)

Runs after section 5 (`summarize.py` template step). Writes `executive_summary_llm.md` without replacing `executive_summary.md`.

**Reference:** [AMD AI Agents 101 — vLLM + MCP workshop notebook](https://github.com/sarimahsan/amd-ai-learning/blob/main/AI%20Agents%20101%20Building%20AI%20Agents%20with%20MCP%20and%20Open-Source%20Inference/build_airbnb_agent_mcp.ipynb) (same `BASE_URL` / `OPENAI_API_KEY` pattern).

### 9a. Launch vLLM in a terminal (MI300)

Open a **new terminal** on the GPU node (not this notebook kernel) and run:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
  --served-model-name Qwen3-30B-A3B \
  --api-key abc-123 \
  --port 8000 \
  --trust-remote-code
```

Optional: `watch rocm-smi` in another terminal to monitor GPU use.

### 9b. Run the cell below

Configures `BASE_URL=http://localhost:8000/v1` and `OPENAI_API_KEY=abc-123` like the workshop, verifies `/v1/models`, then calls `summarize.py --backend vllm` with validation guardrails (if your fork has the latest `summarize.py`).

**OpenAI instead of vLLM:** set `SUMMARY_LLM_BACKEND=openai` and `OPENAI_API_KEY` in `.env`.

In [10]:
load_agent_env()

LLM_BACKEND = os.environ.get("SUMMARY_LLM_BACKEND", "vllm")  # vllm | openai
pr_out = OUT / "pr-5572"

if LLM_BACKEND == "vllm":
    VLLM_URL, VLLM_KEY, VLLM_MODEL = configure_vllm_client_env(
        base_url=os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1"),
        api_key=os.environ.get("OPENAI_API_KEY", "abc-123"),
        model=os.environ.get("VLLM_MODEL", "Qwen3-30B-A3B"),
    )
    print("vLLM client config:", VLLM_URL, VLLM_MODEL)
    if not verify_vllm_server(VLLM_URL, VLLM_KEY):
        print("Start vLLM in a terminal (section 9a) then re-run this cell.")
    summary_llm = run_llm_executive_summary(
        pr_out,
        backend="vllm",
        base_url=VLLM_URL,
        model=VLLM_MODEL,
        output_name="executive_summary_llm.md",
    )
else:
  summary_llm = run_llm_executive_summary(
      pr_out,
      backend="openai",
      model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
      output_name="executive_summary_llm.md",
  )

if summary_llm:
    print("\n--- LLM executive summary ---\n")
    print(summary_llm)
else:
    print(
        "No LLM summary produced. For vLLM: complete section 9a, then re-run. "
        "For OpenAI: set OPENAI_API_KEY in .env."
    )

LLM summary: backend=vllm, model=meta/llama-3.1-70b-instruct, url=http://localhost:8000/v1
$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\summarize.py --backend vllm --input C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\out\pr-5572\report.json --output C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\out\pr-5572\executive_summary_llm.md --base-url http://localhost:8000/v1 --model meta/llama-3.1-70b-instruct --validate-llm --fallback-template
usage: summarize.py [-h] [--input INPUT] [--output OUTPUT]
                    [--backend {template,ollama,openai,vllm}] [--model MODEL]
                    [--base-url BASE_URL]
summarize.py: error: unrecognized arguments: --validate-llm --fallback-template

LLM summary failed (exit 2). Template executive_summary.md from summarize step is unchanged.
No LLM summary produced. For vLLM, start the server and set VLLM_BASE_URL. For 

## 10. Demo checklist

- [ ] `pytest` green
- [ ] PR #5572 → `test:miopen`, `test_filter:quick`, topology warnings show (none expected)
- [ ] Section 9 → `executive_summary_llm.md` when vLLM/OpenAI available
- [ ] PR #5718 → component-scoped `test:*` (needs `GITHUB_TOKEN` for full list)
- [ ] PR #5480 → `test_filter:quick` (third-party packaging)
- [ ] Executive summary + HTML include **Topology warnings** section
- [ ] Open `out/pr-*/report.html` in browser
- [ ] Fork Actions: **Change Impact Upstream PR Scan** workflow dispatch